In [75]:
from transformers import TrainerCallback
from lightning.pytorch.callbacks import Callback
import os
import json
import torch
import numpy as np
import wandb
import pandas as pd
from datetime import datetime
from utils.countdown_utils import *
from tqdm import trange
from pathlib import Path
from transformers import AutoConfig, AutoModelForCausalLM
from litgpt.scripts.convert_lit_checkpoint import convert_lit_checkpoint
from litgpt.utils import copy_config_files, auto_download_checkpoint
%load_ext autoreload
%autoreload 2
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [76]:
def eval_ll(
        model,
        tokenizer,
        data,
        batch_size=128,
        context_len=4096,
        temperature=0.0,
        n=1,
    ):
        """
        Evaluate the model on the data using a sliding window so that the context length is not exceeded
        """
        output_texts_concat = []
        for b in trange(0, len(data), batch_size):
            batch = data[b : min(b + batch_size, len(data))]
            output_texts = ["" for _ in range(len(batch))]
            tokenizer.padding_side = "left"
            inputs = tokenizer(batch, return_tensors="pt", padding=True).to("cuda")
            inputs = inputs["input_ids"]

            if n == 1:
                outputs = model.generate(
                    input_ids=inputs,
                    pad_token_id=tokenizer.eos_token_id,
                    attention_mask=torch.ones_like(inputs),
                    max_length=context_len,
                    num_beams=1,
                    do_sample=False,
                )
                output_tokens = outputs
                output_text = tokenizer.batch_decode(
                    output_tokens, skip_special_tokens=False
                )
                tokenizer.padding_side = "left"
                output_texts = [
                    ot + ot_now for ot, ot_now in zip(output_texts, output_text)
                ]
                output_texts_concat += output_texts

        return output_texts_concat

In [77]:
# data_file = "data/generalization_data/test1_b3_t30_n1000_dfs_filtered.json"
data_file = "data/sos_filtered/test1_b4_t30_n2000000_dfs_filtered.json"

data = []

with open(data_file, "r") as f:
    for line in f:
        if line.strip():  # Skip empty lines
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error parsing line: {e}")
                continue

In [78]:
model_dir = Path("temp/hf_Pythia-6-4-64-dfs-cosine-2")
state_dict = torch.load(model_dir / "model.pth")

hf_model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.bfloat16,
        local_files_only=True,
        state_dict=state_dict,
        attn_implementation="flash_attention_2",
    )

hf_model.cuda()
hf_model.eval()

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)

/tmp/ipykernel_1683198/1892427766.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_dir / "model.pth")


In [79]:
test_prompts = [
    tokenizer.bos_token
    + f"S {sample['target']} [ {' '.join(map(str,sample['nums']))} ] ,"
    for sample in data[: 128]
    ]

In [80]:
predictions = eval_ll(
                hf_model,
                tokenizer,
                data=test_prompts,
                batch_size=128,
                context_len=4096,
                temperature=0.0,
                n=1,
            )

100%|██████████| 1/1 [00:22<00:00, 22.36s/it]


In [81]:
pred_ratings = []
true_rating = []
pred_reasons = []

for i in range(len(predictions)):
    rating, reason = metric_fn(
        predictions[i]
        .split(tokenizer.bos_token)[1]
        .split(tokenizer.eos_token)[0],
        mode="sft",
    )
    tr, _ = metric_fn(f"{data[i]['search_path']}", mode="sft")
    pred_ratings.append(rating)
    true_rating.append(tr)
    pred_reasons.append(reason)

pred_ratings = np.array(pred_ratings)
avg_rating = float(np.mean(pred_ratings))
avg_true_rating = float(np.mean(true_rating))
accuracy = float(np.mean([r > 0 for r in pred_ratings]))
true_accuracy = float(np.mean([r > 0 for r in true_rating]))

In [82]:
pred_ratings

array([0.86892361, 0.86805556, 0.86631944, 0.86805556, 0.86805556,
       0.87239583, 0.8671875 , 0.8671875 , 0.86979167, 0.86371528,
       0.8671875 , 0.86631944, 0.86805556, 0.8671875 , 0.        ,
       0.86545139, 0.86197917, 0.86631944, 0.86631944, 0.86805556,
       0.8671875 , 0.        , 0.86371528, 0.86892361, 0.8671875 ,
       0.87065972, 0.        , 0.86979167, 0.86111111, 0.86284722,
       0.86631944, 0.86458333, 0.        , 0.8671875 , 0.87152778,
       0.86111111, 0.86979167, 0.8671875 , 0.86631944, 0.86979167,
       0.8671875 , 0.86197917, 0.86805556, 0.87152778, 0.87239583,
       0.86892361, 0.86892361, 0.86545139, 0.85850694, 0.86458333,
       0.86805556, 0.86631944, 0.87413194, 0.86805556, 0.8671875 ,
       0.86371528, 0.86805556, 0.86979167, 0.86631944, 0.        ,
       0.86284722, 0.86805556, 0.86979167, 0.86545139, 0.87152778,
       0.86545139, 0.        , 0.86805556, 0.86545139, 0.86805556,
       0.85677083, 0.87065972, 0.86805556, 0.86892361, 0.86892

In [83]:
print(f"Average rating: {avg_rating}")
print(f"-- Average true rating: {avg_true_rating}")
print(f"Accuracy: {accuracy}")
print(f"-- True accuracy: {true_accuracy}")

Average rating: 0.7994452582465278
-- Average true rating: 0.9671088324652778
Accuracy: 0.921875
-- True accuracy: 0.9921875
